In [1]:
import hashlib
import os
import platform
import uuid
import sys
# 修改为自己的学号
STUDENT_ID = 20244080129
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")
MACHINE_FP = hashlib.sha256(
 f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号] {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID] {RUN_ID}")
print(f"[Python] {sys.version.split()[0]}")

[Conda 环境] exp01
[学号] 20244080129
[机器指纹] eff28411a62a643f
[RUN_ID] 20244080129-eff28411a62a643f
[Python] 3.10.22


re 是 Python 自带正则模块，直接 import re，实验主要用 re.findall(pattern, string) 返回所有匹配列表；模式前加 r 写原始字符串。\d 匹配数字，\s 匹配空白，\b 是单词边界，所以 \bcat\b 只匹配独立的 cat，不匹配 cats 里的 cat。量词：* 是 0 次或多次，+ 是 1 次或多次，? 是 0 或 1 次，{n} 恰好 n 次，{n,m} 是 n 到 m 次，因此 ca*t 匹配 ct/cat/caat，ca+t 不匹配 ct，ca?t 只匹配 ct 和 cat，ca{2}t 只匹配 caat，ca{1,2}t 匹配 cat 和 caat。交替 | 从左到右先匹配到就采用，所以 a|aa|aaa 在 “aaaa” 上得四个 a，aaa|aa|a 得 aaa 和 a。任务 1 用 \d+、\s+、\bcat\b 找数字、空白、独立 cat；任务 2 用 cat|dog 演示大小写敏感和交替顺序影响；任务 3 用五个量词对 “ct cat caat caaat” 匹配，对照量词含义即可解释结果。


In [2]:
import re

# 任务 1
text1 = "cat on mat; later: cats and dogs"
print("任务1 文本:", repr(text1))
print("\\d+      :", re.findall(r"\d+", text1))
print("\\s+      :", re.findall(r"\s+", text1))
print("\\bcat\\b :", re.findall(r"\bcat\b", text1))

# 任务 2
text2 = "cat dog bird CAT"
print("\n任务2 文本:", repr(text2))
print("cat|dog  :", re.findall(r"cat|dog", text2))
print('a|aa|aaa on "aaaa" :', re.findall(r"a|aa|aaa", "aaaa"))
print('aaa|aa|a on "aaaa" :', re.findall(r"aaa|aa|a", "aaaa"))

# 任务 3
text3 = "ct cat caat caaat"
print("\n任务3 文本:", repr(text3))
print("ca*t    :", re.findall(r"ca*t", text3))
print("ca+t    :", re.findall(r"ca+t", text3))
print("ca?t    :", re.findall(r"ca?t", text3))
print("ca{2}t  :", re.findall(r"ca{2}t", text3))
print("ca{1,2}t:", re.findall(r"ca{1,2}t", text3))

任务1 文本: 'cat on mat; later: cats and dogs'
\d+      : []
\s+      : [' ', ' ', ' ', ' ', ' ', ' ']
\bcat\b : ['cat']

任务2 文本: 'cat dog bird CAT'
cat|dog  : ['cat', 'dog']
a|aa|aaa on "aaaa" : ['a', 'a', 'a', 'a']
aaa|aa|a on "aaaa" : ['aaa', 'a']

任务3 文本: 'ct cat caat caaat'
ca*t    : ['ct', 'cat', 'caat', 'caaat']
ca+t    : ['cat', 'caat', 'caaat']
ca?t    : ['ct', 'cat']
ca{2}t  : ['caat']
ca{1,2}t: ['cat', 'caat']


本次四个任务都围绕 re 的捕获组和替换/引用展开。任务 1 用 re.findall 配合两组括号分别捕获邮箱的用户名和域名，返回元组列表；任务 2 用后向引用 \1 表示“与前面捕获组完全相同的文本”，从而只匹配连续重复的同一个单词（cat cat、bird bird），而 dog cat 因前后不同不匹配；任务 3 用 re.sub 加三个捕获组分别抓年、月、日，替换串写成 \2-\3-\1 实现“月-日-年”，re.sub 只替换匹配段、保留换行符，所以行数不变；任务 4 同样用 re.sub，捕获区号、中间三位、后四位，替换串 (\1) \2-\3 把区号加括号、其余用连字符连接，括号在替换串中是普通字符无需转义。四个任务的共同点是：先想清楚要捕获哪几段、按什么顺序引用，捕获组编号 \1、\2、\3 与括号从左到右一一对应，替换顺序写错（如任务 3 写成 \2-\1-\3）就会得到错误格式。

In [3]:
import re

# ============ 任务 1：捕获组提取邮箱用户名和域名 ============
text1 = "联系：alice@example.com；备选：bob@test.org"
pattern1 = r"([\w.+-]+)@([\w.-]+\.[a-zA-Z]{2,})"
matches1 = re.findall(pattern1, text1)

print("=" * 50)
print("任务1：提取邮箱用户名和域名")
print("=" * 50)
print("原文本：", text1)
print("匹配结果：", matches1)
for username, domain in matches1:
    print(f"用户名: {username:10s} 域名: {domain}")


# ============ 任务 2：后向引用找连续重复单词 ============
text2 = "cat cat, dog cat, bird bird"
pattern2 = r"\b(\w+)\s+\1\b"
matches2 = re.findall(pattern2, text2)

print("\n" + "=" * 50)
print("任务2：后向引用找连续重复单词")
print("=" * 50)
print("原文本：", text2)
print("连续重复的单词：", matches2)


# ============ 任务 3：日期格式转换 年-月-日 → 月-日-年 ============
text3 = "2024-05-01\n2024/05/02\n2024.05.03"
pattern3 = r"(\d{4})[-/.](\d{2})[-/.](\d{2})"
# \1=年, \2=月, \3=日；要“月-日-年”所以写 \2-\3-\1
converted3 = re.sub(pattern3, r"\2-\3-\1", text3)

print("\n" + "=" * 50)
print("任务3：日期格式转换（年-月-日 → 月-日-年）")
print("=" * 50)
print("转换前：")
print(text3)
print("转换后：")
print(converted3)
print("行数是否保留：", len(text3.splitlines()), "→", len(converted3.splitlines()))


# ============ 任务 4：电话号码统一格式 ============
text4 = "联系我：123-456-7890 或 987.654.3210"
pattern4 = r"(\d{3})[-.](\d{3})[-.](\d{4})"
# \1=区号, \2=中间三位, \3=后四位
converted4 = re.sub(pattern4, r"(\1) \2-\3", text4)

print("\n" + "=" * 50)
print("任务4：电话号码统一为 (123) 456-7890 形式")
print("=" * 50)
print("转换前：", text4)
print("转换后：", converted4)

任务1：提取邮箱用户名和域名
原文本： 联系：alice@example.com；备选：bob@test.org
匹配结果： [('alice', 'example.com'), ('bob', 'test.org')]
用户名: alice      域名: example.com
用户名: bob        域名: test.org

任务2：后向引用找连续重复单词
原文本： cat cat, dog cat, bird bird
连续重复的单词： ['cat', 'bird']

任务3：日期格式转换（年-月-日 → 月-日-年）
转换前：
2024-05-01
2024/05/02
2024.05.03
转换后：
05-01-2024
05-02-2024
05-03-2024
行数是否保留： 3 → 3

任务4：电话号码统一为 (123) 456-7890 形式
转换前： 联系我：123-456-7890 或 987.654.3210
转换后： 联系我：(123) 456-7890 或 (987) 654-3210


In [1]:
%pip install beautifulsoup4 -i https://pypi.tuna.tsinghua.edu.cn/simple

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple

   ---------------------------------------- 0/2 [soupsieve]
   -------------------- ------------------- 1/2 [beautifulsoup4]
   -------------------- ------------------- 1/2 [beautifulsoup4]
   ---------------------------------------- 2/2 [beautifulsoup4]

Note: you may need to restart the kernel to use updated packages.


首先，通过 BeautifulSoup(html_doc, "html.parser") 将字符串转化为解析树，利用 .title、.h1 等标签属性直接定位节点，并通过 .name 和 .get_text() 分别获取标签名和内部文本。其次，find_all() 支持带条件查找，例如用 class_="course" 找到所有指定类名的段落，用 a 找到所有链接，并将其视为列表进行遍历，提取链接的文本及 href 属性（用字典方式 a['href'] 获取）。最后，对比 get_text() 的提取结果可以看出：对单个标签（如段落、链接）使用时，只提取该节点内部的纯文本，剔除 HTML 标签；而对整个文档 soup.get_text() 使用时，则是将全篇所有可见文本拼接在一起，并自动保留原 HTML 代码里的换行与缩进空白，适合快速获取网页正文的原始概貌。

In [3]:
from bs4 import BeautifulSoup

html_doc = """
<html><head><title>实验课程</title></head><body>
<h1>数据分析</h1>
<p class="course">第六讲：正则表达式</p>
<p class="course">第七讲：结构化数据</p>
<a href="https://example.org/lecture06">讲义六</a>
<a href="https://example.org/lecture07">讲义七</a>
</body></html>
"""

# 构建解析树
soup = BeautifulSoup(html_doc, "html.parser")

# ============ 任务 1：标题文本、h1 标签名与文本 ============
print("=" * 50)
print("任务1：标题与 h1")
print("=" * 50)
title = soup.title
print("页面标题文本：", title.get_text())
h1 = soup.h1
print("h1 标签名：", h1.name)
print("h1 文本：", h1.get_text())

# ============ 任务 2：find_all 找 course 段落和链接 ============
print("\n" + "=" * 50)
print("任务2：find_all 查找")
print("=" * 50)

# 所有 class="course" 的段落
courses = soup.find_all("p", class_="course")
print(f"共找到 {len(courses)} 个 class='course' 的段落：")
for i, p in enumerate(courses, 1):
    print(f"  段落{i}: {p.get_text()}")

# 所有链接
links = soup.find_all("a")
print(f"\n共找到 {len(links)} 个链接：")
for i, a in enumerate(links, 1):
    print(f"  链接{i}: 文字='{a.get_text()}', href='{a['href']}'")

# ============ 任务 3：get_text() 对比 ============
print("\n" + "=" * 50)
print("任务3：get_text() 文本提取对比")
print("=" * 50)

print("\n【整个文档的可见文本】")
print(repr(soup.get_text()))
print("【格式化显示】")
print(soup.get_text())

print("\n【单个段落的文本】")
print(repr(courses[0].get_text()))

print("\n【单个链接的文本】")
print(repr(links[0].get_text()))

任务1：标题与 h1
页面标题文本： 实验课程
h1 标签名： h1
h1 文本： 数据分析

任务2：find_all 查找
共找到 2 个 class='course' 的段落：
  段落1: 第六讲：正则表达式
  段落2: 第七讲：结构化数据

共找到 2 个链接：
  链接1: 文字='讲义六', href='https://example.org/lecture06'
  链接2: 文字='讲义七', href='https://example.org/lecture07'

任务3：get_text() 文本提取对比

【整个文档的可见文本】
'\n实验课程\n数据分析\n第六讲：正则表达式\n第七讲：结构化数据\n讲义六\n讲义七\n\n'
【格式化显示】

实验课程
数据分析
第六讲：正则表达式
第七讲：结构化数据
讲义六
讲义七



【单个段落的文本】
'第六讲：正则表达式'

【单个链接的文本】
'讲义六'
